# Few-shot clinical NER on Colab
Select Runtime → Change runtime type → GPU. Upload the updated project ZIP when prompted. Read README.md for interpretation. No new scores are included until you run this notebook.

In [ ]:
import torch
assert torch.cuda.is_available(), "Select a GPU runtime first"
print(torch.cuda.get_device_name(0))

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
uploaded = files.upload()
archive = next(n for n in uploaded if n.endswith(".zip"))
root = Path("/content/clinical_project")
root.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    for entry in z.infolist():
        assert (root / entry.filename).resolve().is_relative_to(root.resolve()), "Unsafe ZIP path"
    z.extractall(root)
projects = list(root.rglob("pyproject.toml"))
assert len(projects) == 1
project = projects[0].parent
%cd {project}

In [ ]:
%pip install -q -e . pytest
!python -m pytest tests -q

## Smoke check
Two episodes to verify the model pipeline. Do not report these as final scores.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable,"scripts/run_pipeline.py","model.device=cuda","data.shot_sizes=[5]","support.n_episodes=2","data.query_examples_per_type=10","output.results_dir=results/smoke"],check=True)

## Dev search and final test
Set INCLUDE_MEDICAL before running. False is cheaper; True adds the suggested medical model. Use a fresh OUTPUT directory for a repeat. Do not choose models using test scores.

In [ ]:
INCLUDE_MEDICAL = False
OUTPUT = "results/experiment"
cmd = [sys.executable,"scripts/tune_and_evaluate.py","--device","cuda","--output",OUTPUT]
if INCLUDE_MEDICAL: cmd.append("--medical")
subprocess.run(cmd,check=True)

In [ ]:
from IPython.display import display, Image
import json
print(json.dumps(json.loads(Path(OUTPUT,"selection.json").read_text()),indent=2))
display(Image(filename=str(Path(OUTPUT,"test/plots/f1_vs_shots.png"))))
display(Image(filename=str(Path(OUTPUT,"test/plots/ece_vs_shots.png"))))

## Optional native GLiNER extraction
Separate exact-span NER benchmark; this does not measure Gaussian prototypes. Label/threshold/span-cap selection uses dev only.

In [ ]:
RUN_EXTRACTION = False
EXTRACTION_MODEL = "urchade/gliner_medium-v2.1"
if RUN_EXTRACTION:
    subprocess.run([sys.executable,"scripts/run_extraction.py","--model",EXTRACTION_MODEL,"--output","results/extraction_dev"],check=True)
    subprocess.run([sys.executable,"scripts/run_extraction.py","--model",EXTRACTION_MODEL,"--split","test","--selection","results/extraction_dev/selection.json","--output","results/extraction_test"],check=True)

## Download fresh results
Do this before disconnecting. Colab storage is temporary. The archive excludes the old dataset results and the embedding cache.

In [ ]:
with open("results/environment.txt","w") as f:
    subprocess.run([sys.executable,"-m","pip","freeze"],stdout=f,check=True)
with zipfile.ZipFile("/content/clinical_results.zip","w",zipfile.ZIP_DEFLATED) as z:
    for path in Path("results").rglob("*"):
        if path.is_file() and "legacy_previous_dataset" not in path.parts:
            z.write(path,str(path))
files.download("/content/clinical_results.zip")